# RWKV-MedSAM2 without distillation — clean launch

Run All to launch training. Dependencies and `/data/loaders32.pkl` must already exist. This copy omits `git pull`, package installation, the embedded TensorBoard server, and all crash/replay diagnostic cells.


# Training

## Setup

### Configuration, installing dependencies, and loading necessary files

In [ ]:
cd /RWKV-MedSAM2

In [ ]:
%load_ext autoreload
%autoreload 2
%env PYTHONWARNINGS=ignore:The value of the smallest subnormal:UserWarning:numpy.core.getlimits

In [ ]:
import os

# OS-level determinism
os.environ["PYTHONHASHSEED"] = "42"
# cuBLAS determinism
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

# Global run seed
RUN_SEED = 42
os.environ["RUN_SEED"] = str(RUN_SEED)

In [ ]:
import torch

if torch.cuda.is_available():
    archs = sorted({f"{torch.cuda.get_device_capability(i)[0]}.{torch.cuda.get_device_capability(i)[1]}"
                    for i in range(torch.cuda.device_count())})
    os.environ["TORCH_CUDA_ARCH_LIST"] = ";".join(archs)  # e.g. "8.0;8.6;8.9"
else:
    os.environ["TORCH_CUDA_ARCH_LIST"] = ""

import numpy as np
    
from rwkv_medsam2.train_sam2 import *

from tqdm.auto import tqdm

In [ ]:
# Load and override configuration for quick sanity check
config_path = "ext/sam2/configs/sam2.1/sam2.1_vcr_nodistill.yaml"
config = load_config(config_path)

# --- Task-block regimen toggles (for complete training) ---
# block_size = 1 disables task blocks (baseline)
config.sampler.task_block_size = 16
config.sampler.task_block_shuffle_tasks = True
config.sampler.task_block_shuffle_within_task = True

# Make epoch size explicit for fair comparisons (optional, but useful)
# config.training.num_samples_per_epoch = 600
# config.training.min_per_bucket = 4

# Decide device manually
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Enable TF32 globally on Ampere+ once
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

In [ ]:
import random, numpy as np

# Prefer config.training.seed if present; otherwise use RUN_SEED
RUN_SEED = int(getattr(getattr(globals().get("config", None), "training", {}), "seed",
                       os.environ.get("RUN_SEED", "42")))

# Seed all RNGs
random.seed(RUN_SEED)
np.random.seed(RUN_SEED)
torch.manual_seed(RUN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RUN_SEED)

# Deterministic kernels
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
    torch.backends.cuda.matmul.allow_tf32 = False
if hasattr(torch.backends, "cudnn") and hasattr(torch.backends.cudnn, "allow_tf32"):
    torch.backends.cudnn.allow_tf32 = False

# DataLoader worker seeding
def seed_worker(worker_id):
    base_seed = torch.initial_seed() % 2**32
    random.seed(base_seed + worker_id)
    np.random.seed(base_seed + worker_id)
    torch.manual_seed(base_seed + worker_id)

# Torch Generator for sampler/shuffle control
def make_generator(seed=RUN_SEED):
    g = torch.Generator()
    g.manual_seed(int(seed))
    return g

In [ ]:
log_cfg = config.logging
logger = setup_logger(log_cfg)
rebuild_loaders = False

#### Build DataLoaders

In [ ]:
import time

if rebuild_loaders:
    print("[data] Rebuilding data loaders...")
    t0 = time.time()
    train_loader, val_loader, test_loader = get_data_loaders(config)
    dt = time.time() - t0
    print(
        f"[data] Done in {dt:.1f}s. "
        f"Train batches: {len(train_loader)}, "
        f"Val batches: {len(val_loader)}, "
        f"Test batches: {len(test_loader)}"
    )

In [ ]:
if rebuild_loaders:
    for split_name, loader in [
        ("TRAIN", train_loader),
        ("VAL",   val_loader),
        ("TEST",  test_loader)
    ]:
        n_pairs   = len(loader.dataset)  # number of pairings
        n_batches = len(loader)          # number of batches
        print(f"{split_name:5s} -> {n_pairs:5d} pairings, {n_batches:4d} batches")

In [ ]:
if rebuild_loaders:
    from rwkv_medsam2.train_sam2 import save_data_loaders, load_data_loaders
    
    save_data_loaders("/data/loaders32.pkl", train_loader, val_loader, test_loader)

#### Load DataLoaders

In [ ]:
train_loader, val_loader, test_loader = load_data_loaders("/data/loaders32.pkl")
print(f"Train loader: {len(train_loader)}, Val loader: {len(val_loader)}, Test loader: {len(test_loader)}")

In [ ]:
# --- EXCLUDE ONE DATASET FROM ALL SPLITS (TRAIN/VAL/TEST) -------------------
from collections import Counter

EXCLUDED_DATASET = "wanglab__CT_DeepLesion-MedSAM2"

def _norm_str(x):
    return str(x).strip() if x is not None else ""

def exclude_sequences_by_dataset(seqs, excluded_dataset, case_insensitive=False):
    ed = _norm_str(excluded_dataset)
    if case_insensitive:
        ed = ed.lower()

    kept = []
    removed = 0
    for s in seqs:
        ds = _norm_str(s.get("dataset"))
        if case_insensitive:
            ds = ds.lower()
        if ds == ed:
            removed += 1
            continue
        kept.append(s)
    return kept, removed

def apply_exclusion_to_loader(loader, excluded_dataset):
    before = len(loader.dataset.sequences)
    new_seqs, removed = exclude_sequences_by_dataset(loader.dataset.sequences, excluded_dataset, case_insensitive=False)
    loader.dataset.sequences = new_seqs
    loader.dataset.entry_dims = [s["dim"] for s in new_seqs]
    after = len(loader.dataset.sequences)
    return before, after, removed

# Before summary
print("\n[EXCLUDE] Before counts (TRAIN/VAL/TEST):")
for name, loader in [("TRAIN", train_loader), ("VAL", val_loader), ("TEST", test_loader)]:
    c = Counter(_norm_str(s.get("dataset")) for s in loader.dataset.sequences)
    print(name, "unique datasets =", len(c), "| excluded_count =", c.get(EXCLUDED_DATASET, 0))

# Apply exclusion
bt, at, rt = apply_exclusion_to_loader(train_loader, EXCLUDED_DATASET)
bv, av, rv = apply_exclusion_to_loader(val_loader, EXCLUDED_DATASET)
bq, aq, rq = apply_exclusion_to_loader(test_loader, EXCLUDED_DATASET)

print(f"\n[EXCLUDE] Removed dataset='{EXCLUDED_DATASET}'")
print(f"  TRAIN: {bt} -> {at}  (removed {rt})")
print(f"  VAL:   {bv} -> {av}  (removed {rv})")
print(f"  TEST:  {bq} -> {aq}  (removed {rq})")

# After summary
print("\n[EXCLUDE] After counts (TRAIN/VAL/TEST):")
for name, loader in [("TRAIN", train_loader), ("VAL", val_loader), ("TEST", test_loader)]:
    c = Counter(_norm_str(s.get("dataset")) for s in loader.dataset.sequences)
    print(name, "unique datasets =", len(c), "| excluded_count =", c.get(EXCLUDED_DATASET, 0))
# ---------------------------------------------------------------------------

In [ ]:
# --- Restrict to ONE dataset (and optional subdataset) -----------------------
from collections import Counter
import copy

def _safe_str(x):
    return str(x) if x is not None else ""

def summarize_dataset_pool(seqs, title="POOL"):
    c = Counter((s.get("dataset"), s.get("subdataset")) for s in seqs)
    print(f"\n[{title}] unique (dataset, subdataset) = {len(c)}")
    for (ds, sd), n in c.most_common(30):
        print(f"  dataset={_safe_str(ds):30s}  subdataset={_safe_str(sd):25s}  n={n}")

def _norm(x):
    # normalize for matching
    return str(x).strip()

def filter_sequences_by_dataset(seqs, target_dataset, target_subdataset=None, case_insensitive=False):
    td = _norm(target_dataset)
    ts = _norm(target_subdataset) if target_subdataset is not None else None
    if case_insensitive:
        td = td.lower()
        ts = ts.lower() if ts is not None else None

    out = []
    for s in seqs:
        ds = _norm(s.get("dataset"))
        sd = _norm(s.get("subdataset"))
        if case_insensitive:
            ds = ds.lower()
            sd = sd.lower()

        if ds != td:
            continue
        if ts is not None and sd != ts:
            continue
        out.append(s)
    return out

# 0) Inspect what you actually have (run once, then set TARGET_* accordingly)
summarize_dataset_pool(train_loader.dataset.sequences, title="TRAIN (before filter)")
summarize_dataset_pool(val_loader.dataset.sequences,   title="VAL   (before filter)")
summarize_dataset_pool(test_loader.dataset.sequences,  title="TEST  (before filter)")

# 1) Set these two lines to the dataset you want
#TARGET_DATASET    = "wanglab__CT_DeepLesion-MedSAM2"
#TARGET_SUBDATASET = "default"

# 2) Filter sequences in-place (keeps everything else the same)
#train_seqs_f = filter_sequences_by_dataset(train_loader.dataset.sequences, TARGET_DATASET, TARGET_SUBDATASET)
#val_seqs_f   = filter_sequences_by_dataset(val_loader.dataset.sequences,   TARGET_DATASET, TARGET_SUBDATASET)
#test_seqs_f  = filter_sequences_by_dataset(test_loader.dataset.sequences,  TARGET_DATASET, TARGET_SUBDATASET)

#train_loader.dataset.sequences  = train_seqs_f
#train_loader.dataset.entry_dims = [s["dim"] for s in train_seqs_f]

#val_loader.dataset.sequences  = val_seqs_f
#val_loader.dataset.entry_dims = [s["dim"] for s in val_seqs_f]

#test_loader.dataset.sequences  = test_seqs_f
#test_loader.dataset.entry_dims = [s["dim"] for s in test_seqs_f]

#print(f"\n[AFTER FILTER] train={len(train_seqs_f)}  val={len(val_seqs_f)}  test={len(test_seqs_f)}")

# 3) Optional but recommended: cap your epoch sizes so you don't request more than exists
# (You can delete these lines if you prefer manual control elsewhere.)
#config.training.num_samples_per_epoch = min(int(getattr(config.training, "num_samples_per_epoch", 600)), len(train_seqs_f))
# ----------------------------------------------------------------------------

#### Setup Hydra

In [ ]:
from hydra import initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

GlobalHydra.instance().clear()
CONFIG_PATH = "/RWKV-MedSAM2/ext/sam2/configs"
initialize_config_dir(config_dir=CONFIG_PATH, version_base=None)

#### Performance

In [ ]:
if torch.cuda.is_available():
    # 1) Autotune convolution algorithms for current shapes
    torch.backends.cudnn.benchmark = True

    # 2) TF32 where safe
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:  # Ampere+
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    # 3) Prefer kernels tuned for FP32 matmuls in PyTorch 2.x
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

### Build teacher and student models

In [ ]:
# Build models and optimizers
student = build_student_predictor(config)
teacher = build_teacher_predictor(config) if distillation_enabled(config) else None

student.to(device)
if teacher is not None:
    teacher.to(device)

student.to(memory_format=torch.channels_last)
if teacher is not None:
    teacher.to(memory_format=torch.channels_last)

### Setup 2D and 3D optimizers and schedulers. Also setup scaler

In [ ]:
# Optimizer + scheduler
opt_all = setup_optimizer_and_scheduler(student, config)
optimizer, scheduler = opt_all["optimizer"], opt_all["scheduler"]

scaler = None

### Enable Multi-GPU support

In [ ]:
# Multi-GPU setup
if torch.cuda.device_count() > 1:
    logger.info(f"Using {torch.cuda.device_count()} GPUs")
    student = torch.nn.DataParallel(student)
    if teacher is not None:
        teacher = torch.nn.DataParallel(teacher)

## Training loop

### Sampling

In [ ]:
# %%
import os, json, math, hashlib
from collections import Counter
from torch.utils.data import DataLoader, Sampler
from rwkv_medsam2.dataset import BalancedTaskBatchSampler
from rwkv_medsam2.train_sam2 import collate_sam2_batch

tasks_map = json.load(open(config.dripp.tasks_file, "r"))

def filter_tasks_map_to_dataset(tasks_map, target_dataset, target_subdataset=None):
    out = {}
    for tid, info in tasks_map.items():
        ds_map = info.get("datasets", {})
        if target_dataset not in ds_map:
            continue
        if target_subdataset is not None:
            if target_subdataset not in ds_map.get(target_dataset, []):
                continue
        out[tid] = info
    return out

# tasks_map = filter_tasks_map_to_dataset(tasks_map, TARGET_DATASET, TARGET_SUBDATASET)
# print(f"[tasks_map] kept {len(tasks_map)} tasks compatible with dataset={TARGET_DATASET} sub={TARGET_SUBDATASET}")

# ------------------------------------------------------------------
# Start from current sequences first so profiles can use actual dataset size
train_sequences = train_loader.dataset.sequences
val_sequences   = val_loader.dataset.sequences

BLOCKED_TASK_IDS = {
    "", ""
}

def _filter_sequences(seqs, blocked_ids):
    out = []
    blocked_ids = set(str(x) for x in blocked_ids)
    for s in seqs:
        task_ids = set(str(t) for t in s.get("tasks", []))
        if not (task_ids & blocked_ids):
            out.append(s)
    return out

train_sequences = _filter_sequences(train_sequences, BLOCKED_TASK_IDS)
val_sequences   = _filter_sequences(val_sequences,   BLOCKED_TASK_IDS)

train_loader.dataset.sequences  = train_sequences
train_loader.dataset.entry_dims = [e["dim"] for e in train_sequences]
val_loader.dataset.sequences    = val_sequences
val_loader.dataset.entry_dims   = [e["dim"] for e in val_sequences]
# ------------------------------------------------------------------

def count_nonempty_buckets(sequences, tasks_map):
    seen = set()
    for seq in sequences:
        ds = seq["dataset"]
        sd = seq.get("subdataset")
        dim = seq.get("dim")
        for task_id in seq.get("tasks", []):
            info = tasks_map.get(task_id, {})
            if ds in info.get("datasets", {}) and sd in info["datasets"].get(ds, []):
                if dim in (2, 3):
                    seen.add((dim, str(task_id)))
    return len(seen)

# ------------------------------------------------------------------
# Data profiles
PROFILE = "full"  # "smoke", "fast", "confidence", "large_5k", "large_10k", "large_25k", "large_50k", "large_100k", "full"

# Set to None to disable dim balancing
TRAIN_TARGET_DIM_RATIO = None
VAL_TARGET_DIM_RATIO   = None

n_train_total = len(train_sequences)
n_val_total   = len(val_sequences)

PROFILE_SETTINGS = {
    "smoke": {
        "train_min_per_bucket": 2,
        "val_min_per_bucket": 2,
        "train_target_items": 78,
        "val_target_items": 80,
    },
    "fast": {
        "train_min_per_bucket": 3,
        "val_min_per_bucket": 2,
        "train_target_items": 300,
        "val_target_items": 100,
    },
    "confidence": {
        "train_min_per_bucket": 4,
        "val_min_per_bucket": 2,
        "train_target_items": 600,
        "val_target_items": 150,
    },
    "large_2.5k": {
        "train_min_per_bucket": 6,
        "val_min_per_bucket": 2,
        "train_target_items": 2_500,
        "val_target_items": 150,
    },
    "large_5k": {
        "train_min_per_bucket": 8,
        "val_min_per_bucket": 2,
        "train_target_items": 5_000,
        "val_target_items": 150,
    },
    "large_10k": {
        "train_min_per_bucket": 10,
        "val_min_per_bucket": 2,
        "train_target_items": 10_000,
        "val_target_items": 150,
    },
    "large_25k": {
        "train_min_per_bucket": 12,
        "val_min_per_bucket": 2,
        "train_target_items": 25_000,
        "val_target_items": 150,
    },
    "large_50k": {
        "train_min_per_bucket": 16,
        "val_min_per_bucket": 2,
        "train_target_items": 50_000,
        "val_target_items": 150,
    },
    "large_100k": {
        "train_min_per_bucket": 20,
        "val_min_per_bucket": 2,
        "train_target_items": 100_000,
        "val_target_items": 150,
    },
    "full": {
        "train_min_per_bucket": 1,
        "val_min_per_bucket": 1,
        "train_target_items": n_train_total,
        "val_target_items": n_val_total,
    },
}

if PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unknown PROFILE={PROFILE}. Valid: {list(PROFILE_SETTINGS.keys())}")

TRAIN_MIN_PER_BUCKET = PROFILE_SETTINGS[PROFILE]["train_min_per_bucket"]
VAL_MIN_PER_BUCKET   = PROFILE_SETTINGS[PROFILE]["val_min_per_bucket"]
train_target_items   = PROFILE_SETTINGS[PROFILE]["train_target_items"]
val_target_items     = PROFILE_SETTINGS[PROFILE]["val_target_items"]

# Floors and feasibility
n_train_buckets = count_nonempty_buckets(train_sequences, tasks_map)
n_val_buckets   = count_nonempty_buckets(val_sequences,   tasks_map)

train_target_items = max(train_target_items, TRAIN_MIN_PER_BUCKET * n_train_buckets)
train_target_items = min(train_target_items, len(train_sequences))
if train_target_items % int(config.training.batch_size) != 0:
    train_target_items = (train_target_items // int(config.training.batch_size)) * int(config.training.batch_size)

val_target_items = max(val_target_items, VAL_MIN_PER_BUCKET * n_val_buckets)
val_target_items = min(val_target_items, len(val_sequences))

RUN_SEED = int(os.environ.get("RUN_SEED", getattr(getattr(config, "training", {}), "seed", 42)))

train_sampler = BalancedTaskBatchSampler(
    train_sequences,
    tasks_map=tasks_map,
    batch_size=int(config.training.batch_size),
    drop_last=True,
    seed=RUN_SEED,
    num_samples=int(train_target_items),
    min_per_bucket=int(TRAIN_MIN_PER_BUCKET),
    target_dim_ratio=TRAIN_TARGET_DIM_RATIO,
)

temp_val_sampler = BalancedTaskBatchSampler(
    val_sequences,
    tasks_map=tasks_map,
    batch_size=1,
    drop_last=False,
    seed=RUN_SEED,
    num_samples=int(val_target_items),
    min_per_bucket=int(VAL_MIN_PER_BUCKET),
    target_dim_ratio=VAL_TARGET_DIM_RATIO,
)

# Make epoch assignment deterministic for diagnostics / previews
if hasattr(train_sampler, "set_epoch"):
    train_sampler.set_epoch(0)
if hasattr(temp_val_sampler, "set_epoch"):
    temp_val_sampler.set_epoch(0)

frozen_val_batches = list(temp_val_sampler)
print(f"[VAL/FROZEN] Using {len(frozen_val_batches)} fixed validation batches.")

VAL_EXPORT_PATH = f"/data/frozen_val_seed{RUN_SEED}_n{len(frozen_val_batches)}.json"

val_selected = []
for batch in frozen_val_batches:
    idx, task_id = batch[0]
    seq = val_loader.dataset.sequences[int(idx)]
    val_selected.append({
        "dataset": seq.get("dataset"),
        "subdataset": seq.get("subdataset"),
        "dim": seq.get("dim"),
        "task_id": str(task_id),
        "image": seq.get("image") if "image" in seq else None,
        "mask":  seq.get("mask")  if "mask"  in seq else None,
        "sequence": seq.get("sequence", None),
    })

with open(VAL_EXPORT_PATH, "w") as f:
    json.dump(val_selected, f, indent=2)

print("Wrote frozen val selection:", VAL_EXPORT_PATH, "n=", len(val_selected))

class FrozenBatchSampler(Sampler):
    def __init__(self, frozen_batches):
        self.frozen_batches = [list(b) for b in frozen_batches]
    def __iter__(self):
        for b in self.frozen_batches:
            yield b
    def __len__(self):
        return len(self.frozen_batches)

val_batch_sampler = FrozenBatchSampler(frozen_val_batches)

g = make_generator(RUN_SEED)

train_loader = DataLoader(
    train_loader.dataset,
    batch_sampler=train_sampler,
    num_workers=int(config.training.num_workers),
    worker_init_fn=seed_worker,
    generator=g,
    persistent_workers=bool(int(config.training.num_workers) > 0),
    prefetch_factor=2 if int(config.training.num_workers) > 0 else None,
    pin_memory=True,
    collate_fn=collate_sam2_batch,
)

val_loader = DataLoader(
    val_loader.dataset,
    batch_sampler=val_batch_sampler,
    num_workers=0,
    worker_init_fn=None,
    generator=g,
    pin_memory=False,
    collate_fn=collate_sam2_batch,
    persistent_workers=False,
)

def _fp_preview(bs, n_batches=3):
    preview = []
    for b_i, batch in enumerate(bs):
        preview.append([(int(i), str(t)) for (i, t) in batch])
        if b_i + 1 >= n_batches:
            break
    enc = json.dumps(preview).encode("utf-8")
    return hashlib.md5(enc).hexdigest()[:16], preview

fp_train, prev_train = _fp_preview(train_sampler, 3)
fp_val,   prev_val   = _fp_preview(temp_val_sampler, 3)

def _bucket_counts_from_assignment(sampler):
    if hasattr(sampler, "set_epoch"):
        sampler.set_epoch(0)
    pools = sampler._assign_primary_buckets()
    counts = {k: len(v) for k, v in pools.items()}
    return dict(sorted(counts.items()))

train_assigned_counts = _bucket_counts_from_assignment(train_sampler)
val_assigned_counts   = _bucket_counts_from_assignment(temp_val_sampler)

print(f"[PROFILE] {PROFILE}")
print(f"[DATA] train_total={n_train_total}  val_total={n_val_total}")
print(f"[RATIO] train_target_dim_ratio={TRAIN_TARGET_DIM_RATIO}  val_target_dim_ratio={VAL_TARGET_DIM_RATIO}")
print(f"[TRAIN] min_per_bucket={TRAIN_MIN_PER_BUCKET}  num_items={train_target_items}  ~batches={len(train_sampler)}  fp={fp_train}")
print(f"[VAL]   min_per_bucket={VAL_MIN_PER_BUCKET}    num_items={val_target_items}   ~batches={len(temp_val_sampler)}  fp={fp_val}")

print(f"[TRAIN] num_samples target: {train_sampler.num_samples}")
print(f"[TRAIN] effective_target:  {getattr(train_sampler, 'effective_target', None)}")
print(f"[TRAIN] post_dim_effective_target: {getattr(train_sampler, 'post_dim_effective_target', None)}")
print(f"[TRAIN] valid unique indices: {len(getattr(train_sampler, 'valid_indices', []))}")
print(f"[TRAIN] #raw non-empty buckets: {len(getattr(train_sampler, 'raw_bucket_sizes', {}))}")
print("[TRAIN] raw bucket sizes (membership counts, before per-epoch assignment):")
for (dim, task), q in sorted(getattr(train_sampler, "raw_bucket_sizes", {}).items()):
    if q > 0:
        print(f"  dim={dim} task={task:20s} raw_count={q}")
print("Sum raw bucket memberships =", sum(getattr(train_sampler, "raw_bucket_sizes", {}).values()))

print("\n[TRAIN] assigned unique samples per bucket for epoch 0:")
for (dim, task), q in sorted(train_assigned_counts.items()):
    if q > 0:
        print(f"  dim={dim} task={task:20s} assigned={q}")
print("Sum assigned unique samples =", sum(train_assigned_counts.values()))

print(f"\n[VAL/SOURCE] num_samples target: {temp_val_sampler.num_samples}")
print(f"[VAL/SOURCE] effective_target:  {getattr(temp_val_sampler, 'effective_target', None)}")
print(f"[VAL/SOURCE] post_dim_effective_target: {getattr(temp_val_sampler, 'post_dim_effective_target', None)}")
print(f"[VAL/SOURCE] valid unique indices: {len(getattr(temp_val_sampler, 'valid_indices', []))}")
print(f"[VAL/SOURCE] #raw non-empty buckets: {len(getattr(temp_val_sampler, 'raw_bucket_sizes', {}))}")
print("[VAL/SOURCE] raw bucket sizes (membership counts, before per-epoch assignment):")
for (dim, task), q in sorted(getattr(temp_val_sampler, "raw_bucket_sizes", {}).items()):
    if q > 0:
        print(f"  dim={dim} task={task:20s} raw_count={q}")
print("Sum raw bucket memberships =", sum(getattr(temp_val_sampler, "raw_bucket_sizes", {}).values()))

print("\n[VAL/SOURCE] assigned unique samples per bucket for epoch 0:")
for (dim, task), q in sorted(val_assigned_counts.items()):
    if q > 0:
        print(f"  dim={dim} task={task:20s} assigned={q}")
print("Sum assigned unique samples =", sum(val_assigned_counts.values()))

In [ ]:
from pathlib import Path
import json

TASKS_PATH = Path(config.dripp.tasks_file)
print("config.dripp.tasks_file =", str(TASKS_PATH))
print("exists =", TASKS_PATH.exists())
print("size_bytes =", TASKS_PATH.stat().st_size if TASKS_PATH.exists() else None)

raw = TASKS_PATH.read_text()
print("\ncontains 'wanglab__CT_DeepLesion-MedSAM2' =", ("wanglab__CT_DeepLesion-MedSAM2" in raw))
print("contains 'CT_DeepLesion-MedSAM2' =", ("CT_DeepLesion-MedSAM2" in raw))

# show the exact datasets keys inside the task entry, if present
tasks_map_disk = json.loads(raw)
tid = "ct_lesions_deeplesion"
print("\nTask present on disk =", tid in tasks_map_disk)
if tid in tasks_map_disk:
    print("Task datasets keys =", list((tasks_map_disk[tid].get("datasets") or {}).keys()))
    print("Task datasets map  =", tasks_map_disk[tid].get("datasets"))

# sanity: what dataset string is in sequences?
print("\nExample sequence dataset/subdataset:")
print("train:", train_loader.dataset.sequences[0]["dataset"], train_loader.dataset.sequences[0].get("subdataset"))
print("val:  ", val_loader.dataset.sequences[0]["dataset"], val_loader.dataset.sequences[0].get("subdataset"))

In [ ]:
# --- Inspect TRAIN assigned coverage ---
def _assigned_counts_for_epoch0(sampler):
    if hasattr(sampler, "set_epoch"):
        sampler.set_epoch(0)
    pools = sampler._assign_primary_buckets()
    return {k: len(v) for k, v in sorted(pools.items())}

train_assigned_counts = _assigned_counts_for_epoch0(train_sampler)
val_assigned_counts   = _assigned_counts_for_epoch0(temp_val_sampler)

print(f"[TRAIN] num_samples target: {train_sampler.num_samples}")
print(f"[TRAIN] effective_target:  {getattr(train_sampler, 'effective_target', None)}")
print(f"[TRAIN] valid unique indices: {len(getattr(train_sampler, 'valid_indices', []))}")
print(f"[TRAIN] #raw buckets (non-empty): {len(getattr(train_sampler, 'raw_bucket_sizes', {}))}")

print("[TRAIN] raw bucket sizes (before per-epoch assignment):")
for (dim, task), q in sorted(getattr(train_sampler, "raw_bucket_sizes", {}).items()):
    if q > 0:
        print(f"  dim={dim} task={task:20s} raw_count={q}")
print("Sum raw bucket memberships =", sum(getattr(train_sampler, "raw_bucket_sizes", {}).values()))

print("\n[TRAIN] assigned unique samples per bucket for epoch 0:")
for (dim, task), q in sorted(train_assigned_counts.items()):
    if q > 0:
        print(f"  dim={dim} task={task:20s} assigned={q}")
print("Sum assigned unique samples =", sum(train_assigned_counts.values()))

# --- Inspect VAL assigned coverage ---
print(f"\n[VAL] num_samples target: {temp_val_sampler.num_samples}")
print(f"[VAL] effective_target:  {getattr(temp_val_sampler, 'effective_target', None)}")
print(f"[VAL] valid unique indices: {len(getattr(temp_val_sampler, 'valid_indices', []))}")
print(f"[VAL] #raw buckets (non-empty): {len(getattr(temp_val_sampler, 'raw_bucket_sizes', {}))}")

print("[VAL] raw bucket sizes (before per-epoch assignment):")
for (dim, task), q in sorted(getattr(temp_val_sampler, "raw_bucket_sizes", {}).items()):
    if q > 0:
        print(f"  dim={dim} task={task:20s} raw_count={q}")
print("Sum raw bucket memberships =", sum(getattr(temp_val_sampler, "raw_bucket_sizes", {}).values()))

print("\n[VAL] assigned unique samples per bucket for epoch 0:")
for (dim, task), q in sorted(val_assigned_counts.items()):
    if q > 0:
        print(f"  dim={dim} task={task:20s} assigned={q}")
print("Sum assigned unique samples =", sum(val_assigned_counts.values()))

In [ ]:
# === Per-task imaging modality distribution (plain text, TB-style) ===
import os, json, glob
from collections import Counter, defaultdict
from rwkv_medsam2.dataset import BalancedTaskBatchSampler

ALLOWED_MODALITIES = [
    "ct","mri","x-ray","ultrasound","retinal-scan","colonoscopy",
    "fetoscopy","histopathology","dermoscopy","mammography","oct"
]
ALL_MODAL_COLS = ALLOWED_MODALITIES + ["<unknown>"]

def _safe_lower(s):
    return str(s).strip().lower() if s is not None else None

def discover_group_files(candidate_roots):
    found = set()
    for root in candidate_roots:
        if not root or not os.path.isdir(root):
            continue
        for p in glob.glob(os.path.join(root, "*", "*_groups.json")):
            found.add(os.path.abspath(p))
        for p in glob.glob(os.path.join(root, "*_groups.json")):
            found.add(os.path.abspath(p))
    return sorted(found)

def build_modality_map_from_group_files(group_files):
    """Returns dict[(dataset, subdataset)] = modality (lowercased within ALLOWED_MODALITIES)."""
    mod_map = {}
    for fp in group_files:
        try:
            data = json.load(open(fp, "r"))
        except Exception as e:
            print(f"[WARN] Could not read {fp}: {e}")
            continue

        ds_name = _safe_lower(os.path.splitext(os.path.basename(fp))[0].replace("_groups", "")) \
                  or _safe_lower(os.path.basename(os.path.dirname(fp)))

        for sub in data.get("subdatasets", []):
            sub_name = _safe_lower(sub.get("name", "default"))
            modality = sub.get("modality")
            if modality is None:
                for split_key in ("train","val","test"):
                    items = sub.get(split_key, [])
                    if items:
                        modality = items[0].get("subdataset_modality")
                        if modality is not None:
                            break
            modality = _safe_lower(modality)
            if modality in ALLOWED_MODALITIES:
                mod_map[(ds_name, sub_name)] = modality
    return mod_map

def flatten_indices_from_batches(batches):
    """Sampler yields batches containing either idx or (idx, task_id)."""
    flat = []
    for b in batches:
        for it in b:
            idx = it[0] if isinstance(it, (tuple, list)) else it
            flat.append(int(idx))
    return flat

def sequences_iter(sequences, indices=None):
    if indices is None:
        for s in sequences:
            yield s
    else:
        for i in indices:
            yield sequences[int(i)]

def per_task_modality_counts(sequences, modality_map, indices=None):
    """
    Returns:
      counts[(task, modality)] = int
      task_totals[task] = int
      task_list = sorted list of tasks encountered
      overall_modality_totals[modality] = int
    """
    counts = Counter()
    task_totals = Counter()
    overall_mod_totals = Counter()
    seen_tasks = set()

    for s in sequences_iter(sequences, indices):
        ds  = _safe_lower(s.get("dataset"))
        sd  = _safe_lower(s.get("subdataset"))
        mod = modality_map.get((ds, sd), "<unknown>")
        tasks = s.get("tasks", [])
        if not tasks:
            # If a sequence somehow has no tasks, still tally modality under a pseudo-task?
            # We'll just skip; dataset buckets are task-driven in your pipeline.
            continue
        for t in tasks:
            seen_tasks.add(t)
            counts[(t, mod)] += 1
            task_totals[t]   += 1
            overall_mod_totals[mod] += 1

    task_list = sorted(seen_tasks)
    return counts, task_totals, task_list, overall_mod_totals

def print_block_header(title):
    print("\n" + "=" * 80)
    print(f"{title}")
    print("=" * 80)

def print_overall_totals(overall_mod_totals, total_items, indent=""):
    print(f"{indent}OVERALL (by modality): total_items={total_items}")
    for m in ALL_MODAL_COLS:
        c = overall_mod_totals.get(m, 0)
        if c > 0:
            pct = 100.0 * c / max(total_items, 1)
            print(f"{indent}  - {m:16s}: {c:6d}  ({pct:5.1f}%)")

def print_task_table(block_name, counts, task_totals, task_list, indent=""):
    # Header
    head = f"{indent}{'TASK':30s} {'TOTAL':>7s}  " + "  ".join([f"{m[:12]:>12s}" for m in ALL_MODAL_COLS])
    print(head)
    print(f"{indent}{'-'*len(head)}")

    # Rows
    for t in task_list:
        total = task_totals.get(t, 0)
        row_counts = [counts.get((t, m), 0) for m in ALL_MODAL_COLS]
        row = f"{indent}{t:30s} {total:7d}  " + "  ".join([f"{c:12d}" for c in row_counts])
        print(row)

    print("")  # spacer

    # Percent rows
    head_pct = f"{indent}{'TASK (percent)':30s} {'':7s}  " + "  ".join([f"{m[:12]:>12s}" for m in ALL_MODAL_COLS])
    print(head_pct)
    print(f"{indent}{'-'*len(head_pct)}")
    for t in task_list:
        total = max(task_totals.get(t, 0), 1)
        row_pcts = [100.0 * counts.get((t, m), 0) / total for m in ALL_MODAL_COLS]
        row = f"{indent}{t:30s} {'':7s}  " + "  ".join([f"{p:11.1f}%" for p in row_pcts])
        print(row)

def summarize_block(block_title, sequences, modality_map, sampler=None, batch_size=1, drop_last=False, seed=42, num_samples=None, min_per_bucket=1):
    print_block_header(block_title + " — POOL")
    pool_counts, pool_totals, pool_tasks, pool_mod_totals = per_task_modality_counts(sequences, modality_map, indices=None)
    total_items_pool = sum(pool_totals.values())
    print_task_table(block_title, pool_counts, pool_totals, pool_tasks, indent="")
    print_overall_totals(pool_mod_totals, total_items_pool, indent="")

    if sampler is not None:
        print_block_header(block_title + " — SUBSAMPLE")
        preview = BalancedTaskBatchSampler(
            sequences,
            tasks_map=sampler.tasks_map,
            batch_size=batch_size,
            drop_last=drop_last,
            seed=seed,
            num_samples=num_samples if num_samples is not None else sampler.num_samples,
            min_per_bucket=min_per_bucket if hasattr(sampler, "min_per_bucket") else 1,
        )
        batches = list(preview)
        sel_idx = flatten_indices_from_batches(batches)
        sub_counts, sub_totals, sub_tasks, sub_mod_totals = per_task_modality_counts(sequences, modality_map, indices=sel_idx)
        total_items_sub = sum(sub_totals.values())

        print(f"[{block_title}] subsample: batches={len(batches)}  items={len(sel_idx)}")
        print_task_table(block_title, sub_counts, sub_totals, sub_tasks, indent="")
        print_overall_totals(sub_mod_totals, total_items_sub, indent="")

# ---- Discover groups files ----
candidate_roots = []
try:
    candidate_roots.append(str(config.dripp.output_dir))
except Exception:
    pass
candidate_roots.append("/mnt/data")

group_files = discover_group_files(candidate_roots)
if not group_files:
    raise FileNotFoundError("No '*_groups.json' files found under: " + ", ".join(candidate_roots))

print("[INFO] Found groups files:")
for p in group_files:
    print("  -", p)

# ---- Build (dataset, subdataset) -> modality ----
mod_map = build_modality_map_from_group_files(group_files)
print(f"[INFO] Modality map entries: {len(mod_map)}")

# ---- TRAIN ----
summarize_block(
    "TRAIN",
    sequences=train_loader.dataset.sequences,
    modality_map=mod_map,
    sampler=train_sampler,
    batch_size=int(config.training.batch_size),
    drop_last=True,
    seed=int(getattr(config.training, "seed", 42)),
    num_samples=int(train_sampler.num_samples),
    min_per_bucket=int(getattr(train_sampler, "min_per_bucket", 1)),
)

# ---- VAL ----
summarize_block(
    "VAL",
    sequences=val_loader.dataset.sequences,
    modality_map=mod_map,
    sampler=temp_val_sampler,
    batch_size=1,
    drop_last=False,
    seed=int(getattr(config.training, "seed", 42)),
    num_samples=int(temp_val_sampler.num_samples),
    min_per_bucket=int(getattr(temp_val_sampler, "min_per_bucket", 1)),
)


### Setup TensorBoard + Checkpoints

In [ ]:
# TensorBoard + checkpoints
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
%load_ext tensorboard
TB_COMPARE_ROOT = "/data/TrainingLogs/tb_compare"
TB_EXPERIMENT_NAME = "no_distill"
tb_dir = os.path.join(TB_COMPARE_ROOT, TB_EXPERIMENT_NAME)
run_dir = os.path.join(
    tb_dir,
    datetime.now().strftime("%Y%m%d-%H%M%S")
    + f"-seed{config.training.seed}-bs{config.training.batch_size}"
)
os.makedirs(tb_dir, exist_ok=True)
writer  = SummaryWriter(log_dir=run_dir, flush_secs=5, max_queue=10)

TRAIN_TIMING_TAGS = (
    "train/2d_step_time_avg_s",
    "train/3d_step_time_avg_s",
    "train/epoch_time_s",
)
logger.info(f"Training timing metrics will be logged under: {TRAIN_TIMING_TAGS}")

writer.add_hparams({
    "bs":           int(  getattr(config.training, "batch_size",   1)),
    "epochs":       int(  getattr(config.training, "epochs",       1)),
    "lr_backbone":  float(getattr(config.training, "lr_backbone",  getattr(config.training, "lr", 1e-4))),
    "lr_mask":      float(getattr(config.training, "mask_lr",      1e-4)),
    "lr_mem":       float(getattr(config.training, "mem_lr",       1e-4)),
    "weight_decay": float(getattr(config.training, "weight_decay", 0.0)),
    "seed":         int(  getattr(config.training, "seed",         42)),
}, {"val/iou_best": 0.0})

best_d3d    = 0.0
ckpt_mgr    = CheckpointManager(config.ckpt_path)
val_freq    = int(config.training.val_freq)
global_step = 0

### Complete Training

In [ ]:
import os
import torch

def _unwrap_model(m):
    return m.module if hasattr(m, "module") else m

def _select_resume_checkpoint(ckpt_dir):
    latest_path = os.path.join(ckpt_dir, "latest.pth")
    if os.path.exists(latest_path):
        return latest_path

    # Fall back to the highest numbered epoch_*.pth snapshot.
    epoch_paths = []
    for name in os.listdir(ckpt_dir) if os.path.isdir(ckpt_dir) else []:
        if name.startswith("epoch_") and name.endswith(".pth"):
            try:
                epoch_idx = int(name[len("epoch_"):-len(".pth")])
            except ValueError:
                continue
            epoch_paths.append((epoch_idx, os.path.join(ckpt_dir, name)))

    if epoch_paths:
        return max(epoch_paths, key=lambda item: item[0])[1]
    return None

resume_ckpt_path = _select_resume_checkpoint(config.ckpt_path)

best_ckpt_path = os.path.join(config.ckpt_path, "best.pth")

start_epoch = 0

if "global_step" not in globals():
    global_step = 0

if "best_d3d" not in globals():
    best_d3d = float("-inf")

# Resume from the latest training checkpoint if one exists
if resume_ckpt_path is not None:
    ckpt = torch.load(resume_ckpt_path, map_location="cpu")

    _unwrap_model(student).load_state_dict(ckpt["model"], strict=True)

    if ckpt.get("optimizer") is not None:
        optimizer.load_state_dict(ckpt["optimizer"])

    if ckpt.get("scheduler") is not None:
        scheduler.load_state_dict(ckpt["scheduler"])

    if ckpt.get("scaler") is not None and scaler is not None:
        scaler.load_state_dict(ckpt["scaler"])

    start_epoch = int(ckpt.get("epoch", -1)) + 1
    global_step = int(ckpt.get("global_step", 0))

    logger.info(
        f"Resumed from checkpoint: {resume_ckpt_path} | "
        f"last_epoch={ckpt.get('epoch', 'NA')} | "
        f"start_epoch={start_epoch} | global_step={global_step}"
    )
else:
    logger.info("No checkpoint found. Starting from scratch.")

# Restore best metric tracker if possible
if os.path.exists(best_ckpt_path):
    best_ckpt = torch.load(best_ckpt_path, map_location="cpu")
    if best_ckpt.get("metric") is not None:
        best_d3d = float(best_ckpt["metric"])
        logger.info(f"Restored best_d3d={best_d3d:.6f} from {best_ckpt_path}")

# Train epoch
for epoch in range(start_epoch, int(config.training.epochs)):
    if hasattr(train_loader, "batch_sampler") and hasattr(train_loader.batch_sampler, "set_epoch"):
        train_loader.batch_sampler.set_epoch(epoch)
        logger.info(f"[Epoch {epoch}] train_loader.batch_sampler.set_epoch({epoch})")

    train_loss, global_step, optimizer, scheduler = train_epoch(
        student, teacher, train_loader,
        optimizer, scheduler,
        config, scaler, epoch, logger, writer, global_step
    )
    logger.info(f"[Epoch {epoch}] Train loss: {train_loss:.4f}")

    # Always save rolling latest + fixed epoch snapshot
    latest_path = ckpt_mgr.save_latest(
        student, optimizer, scheduler, scaler, epoch, global_step
    )
    epoch_path = ckpt_mgr.save_epoch(
        student, optimizer, scheduler, scaler, epoch, global_step, metric=None
    )

    logger.info(f"[Epoch {epoch}] Saved latest checkpoint: {latest_path}")
    logger.info(f"[Epoch {epoch}] Saved epoch checkpoint: {epoch_path}")

    do_validate = (
        epoch == 0
        or ((epoch + 1) % 5 == 0)
        or (epoch == int(config.training.epochs) - 1)
    )

    if do_validate:
        val_metrics = validate_epoch(student, val_loader, config, epoch, writer)
        val_d3d_best = float(val_metrics["dice3d@best"])

        best_path = ckpt_mgr.save_best(
            student, optimizer, scheduler, scaler, epoch, val_d3d_best, global_step
        )

        if val_d3d_best > best_d3d:
            best_d3d = val_d3d_best

        if best_path is not None:
            logger.info(
                f"[Epoch {epoch}] New best checkpoint saved: {best_path} "
                f"(dice3d@best={val_d3d_best:.6f})"
            )

        logger.info(f"[Epoch {epoch}] Validation metrics: {val_metrics}")

writer.close()
logger.info("Training complete.")